# emg2pose in Google Colab (uv, no Conda)

Select **Runtime → Change runtime type → T4 GPU** for GPU inference; CPU also works.
Run the cells in order. Change the repository URL/branch below to a checkout
containing the uv setup before running this notebook.

The original project's Python dependency versions are preserved. uv creates a
separate Python 3.10 environment, including both original editable installs
(emg2pose and UmeTrack). Project code runs through `run_project(...)` in that
environment, independently of Colab's notebook Python. **Each call is a new
process; Python variables do not persist between calls.** Files do persist.
The existing application code and getting-started notebook are unchanged.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import textwrap

REPO_URL = "https://github.com/eljoserass/emg2pose.git"
REPO_REF = "main"
REPO_DIR = Path("/content/emg2pose")
DATA_DIR = Path("/content/emg2pose-data")
OUTPUT_DIR = Path("/content/emg2pose-outputs")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError("Choose a repository/branch containing the new uv setup.")

# Install uv into its own tool environment, leaving Colab's packages intact.
UV = shutil.which("uv")
if UV is None:
    tool_dir = Path("/content/emg2pose-uv-tool")
    subprocess.run([sys.executable, "-m", "venv", str(tool_dir)], check=True)
    subprocess.run(
        [str(tool_dir / "bin/python"), "-m", "pip", "install", "uv"],
        check=True,
    )
    UV = str(tool_dir / "bin/uv")

gpu_names = ""
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        gpu_names = result.stdout.strip()
TORCH_EXTRA = "cu121" if gpu_names else "cpu"
print(f"Colab Python: {sys.version.split()[0]}; project wheel: {TORCH_EXTRA}")
if gpu_names:
    print(f"GPU: {gpu_names}")

subprocess.run(
    [UV, "sync", "--python", "3.10", "--no-dev", "--extra", TORCH_EXTRA],
    cwd=REPO_DIR, check=True,
)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

project_env = {
    **os.environ,
    "EMG2POSE_DATA_DIR": str(DATA_DIR),
    "EMG2POSE_OUTPUT_DIR": str(OUTPUT_DIR),
    "PYTHONUNBUFFERED": "1",
}

def run_project(code):
    """Execute Python in the project's uv environment; stream its output."""
    return subprocess.run(
        [UV, "run", "--no-sync", "python", "-"],
        input=textwrap.dedent(code), text=True, check=True,
        cwd=REPO_DIR, env=project_env,
    )

run_project("""
import sys
import torch
import emg2pose.visualization
import lib.common.hand
print("Project Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
""")


## Download the mini dataset and pretrained checkpoints

The mini dataset is about 600 MiB; the checkpoint archive is an additional
download. Both come from the original project's published endpoints.
Downloads and extraction use the same directory. Completed downloads are reused.
Everything under `/content` is temporary and disappears when Colab resets.


In [ ]:
import tarfile
from urllib.request import urlretrieve

BASE_URL = "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose"

def download(filename):
    destination = DATA_DIR / filename
    if not destination.exists():
        partial = destination.with_name(destination.name + ".part")
        print(f"Downloading {filename}...")
        urlretrieve(f"{BASE_URL}/{filename}", partial)
        partial.replace(destination)
    return destination

def extract(filename):
    marker = DATA_DIR / (filename + ".extracted")
    if not marker.exists():
        archive = download(filename)
        print(f"Extracting {filename}...")
        with tarfile.open(archive) as bundle:
            bundle.extractall(DATA_DIR, filter="data")
        marker.touch()

download("emg2pose_metadata.csv")
extract("emg2pose_dataset_mini.tar")
extract("emg2pose_model_checkpoints.tar.gz")
sessions = sorted((DATA_DIR / "emg2pose_dataset_mini").glob("*.hdf5"))
if not sessions:
    raise RuntimeError(f"No HDF5 sessions found under {DATA_DIR}.")
print(f"Found {len(sessions)} sessions")


## Run inference and export interactive hand animations

This runs a five-second window with the original tracking checkpoint and saves
predictions as NumPy arrays. HTML plots avoid Kaleido/FFmpeg video rendering
and the original notebook's 32 rendering workers.


In [ ]:
run_project("""
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from emg2pose.data import Emg2PoseSessionData
from emg2pose.lightning import Emg2PoseModule
from emg2pose.utils import downsample
from emg2pose.visualization import get_plotly_animation_for_joint_angles

data_dir = Path(os.environ["EMG2POSE_DATA_DIR"])
output_dir = Path(os.environ["EMG2POSE_OUTPUT_DIR"])
sessions = sorted((data_dir / "emg2pose_dataset_mini").glob("*.hdf5"))
session = Emg2PoseSessionData(str(sessions[min(15, len(sessions) - 1)]))
print(pd.read_csv(data_dir / "emg2pose_metadata.csv").head().to_string(index=False))
print(session)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = data_dir / "emg2pose_model_checkpoints/tracking_vemg2pose.ckpt"
module = Emg2PoseModule.load_from_checkpoint(str(checkpoint), map_location="cpu")
module = module.to(device).eval()

stop = min(10_000, len(session))
window = session[:stop]
batch = {
    "emg": torch.tensor(window["emg"].T[None].copy(), dtype=torch.float32, device=device),
    "joint_angles": torch.tensor(window["joint_angles"].T[None].copy(), dtype=torch.float32, device=device),
    "no_ik_failure": torch.tensor(session.no_ik_failure[:stop][None], device=device),
}
with torch.inference_mode():
    predictions, targets, mask = module(batch)

predictions = predictions[0].T.cpu().numpy()
targets = targets[0].T.cpu().numpy()
np.savez(output_dir / "predictions.npz", predictions=predictions, targets=targets,
         no_ik_failure=mask[0].cpu().numpy())
print("Device:", device, "Predictions:", predictions.shape)

for name, angles, color in [("ground_truth", targets, "gray"),
                             ("predictions", predictions, "lightpink")]:
    angles_30hz = downsample(angles, native_fs=2000, target_fs=30)
    fig = get_plotly_animation_for_joint_angles(angles_30hz[:90], color=color)
    fig.write_html(str(output_dir / f"{name}.html"), include_plotlyjs=True, auto_play=False)
print("Outputs:", output_dir)
""")


In [ ]:
from IPython.display import HTML, display

for name in ("ground_truth", "predictions"):
    print(name.replace("_", " ").title())
    display(HTML((OUTPUT_DIR / f"{name}.html").read_text()))


## Optional: short training run

Set `RUN_TRAINING = True` to train for one epoch with a small batch and a
limited number of batches using the original training entry point.
This checks the training workflow, rather than training a useful model.
Copy the output directory (including Lightning checkpoints) to Google Drive
before ending the runtime if you want to keep it.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [UV, "run", "--no-sync", "python", "-m", "emg2pose.train",
         "train=True", "eval=False", "experiment=tracking_vemg2pose",
         "data_split=mini_split",
         f"data_location={DATA_DIR / 'emg2pose_dataset_mini'}",
         "batch_size=4", "num_workers=0", "trainer.max_epochs=1",
         "+trainer.limit_train_batches=10", "+trainer.limit_val_batches=2",
         f"hydra.run.dir={OUTPUT_DIR / 'training'}"],
        cwd=REPO_DIR, env=project_env, check=True,
    )
